
# Jobs & Pipelines

The Jobs & Pipelines tab allows you to build complex data pipelines by chaining together multiple scripts, queries, notebooks and logic. Each one of these is called a task.

They can be used to build Reproducible Analytical Pipelines (RAP) that can be re-run with different parameters and have all inputs and outputs audited automatically. 

In addition, Jobs and Pipelines can be set to run on a schedule or a trigger you define, and can be setup to notify people when it succeeds (/ fails).

Other recommended uses of jobs and pipelines are any data modelling tasks such as cleaning your source data and collating it into a more analytically friendly format in your modelling area.

The Jobs & Pipelines page can be accessed from the left-hand side bar. 

![./../../../images/jobs_location.png](././../../../images/jobs_location.png "./../../../images/jobs_location.png")

## Below you will find a guide talking you through the creation of a job in two different ways - The Databricks interface and the Databricks API

### You can also create a job with yml code.

To see how to do something in yml we recommend you use one of the below methods to create a pipeline and then click "Edit as YAML" from the dropdown next to the blue "Run now" button in the top left.

![./../../../images/jobs_yaml.png](././../../../images/jobs_yaml.png "./../../../images/jobs_yaml.png")

## Benefits of Jobs

There are some key benefits of jobs:
1. They chain together tasks in a set order, this improves the reproducibility of the task, enables easy identification of failures.
2. Jobs can be scheduled, this both means that they can run automatically without user input and that they can be specified to run at less energy taxing times of the day.
3. Jobs are processes as code, meaning that they can be stored in git accounts, change managed, and migrated easily.
4. Jobs enable you to build in the approval step into the run.

## Cons of Jobs

There are also a few cons to jobs:
1. Once set-up with a schedule a job will run until a user specifies it to stop, this means that energy and money might be wasted.
2. Jobs are a more complex way of running notebooks and code, meaning that they add an extra layer of maintenance that might not be necessary.
3. Code cannot be edited inline as a job runs.
4. Give a false sense of security, jobs can fail just like scripts, and a green tick mark doens't always mean it succeeded.

## Tasks and dependencies

Each step in a Job or Pipeline is referred to as a task and each task can have dependencies, which are other tasks that must have run before it. 

Tasks are usually scripts, or Notebooks which can be held either in your Databricks Workspace, or (preferably) in a Git repository either on Github or Azure DevOps. 

Below is an example of a Job with two tasks. The last task will only run once all other tasks have completed as it is dependant on them having completed.

![./../../../images/jobs_tasks.png](././../../../images/jobs_tasks.png "./../../../images/jobs_tasks.png")

## Auditing

Each time a Job or Pipeline is run, Databricks audits:
- any input parameters
- all outputs
- the success and failure of each task
- when it was run and who by

![./../../../images/jobs_synopsis.png](././../../../images/jobs_synopsis.png "./../../../images/jobs_synopsis.png")
![./../../../images/jobs_run_history.png](././../../../images/jobs_run_history.png "./../../../images/jobs_run_history.png")

This makes Jobs and Pipelines a very powerful debugging tool as you can refer back to results from previous runs. This means that if your pipeline fails you can review the notebook(s) that failed and troubleshoot the issue.

![./../../../images/jobs_fail_history.png](././../../../images/jobs_fail_history.png "./../../../images/jobs_fail_history.png")
![./../../../images/jobs_failure.png](././../../../images/jobs_failure.png "./../../../images/jobs_failure.png")

Jobs and Pipelines that fail also allow you to repair the job or pipeline once you have found and fixed the issue. This prevents having to re-run the whole pipeline from scratch and allows it to pick up from the point where it failed.

## Coded jobs

Another useful aspect of jobs and pipelines is that they can be defined and ran using code through the [DataBricks Jobs API](https://docs.databricks.com/api/workspace/jobs).

There is an R library which has been created to interface with the DataBricks API, meaning that you can script jobs in R using the [DataBricks SDK for R package](https://docs.databricks.com/en/dev-tools/sdk-r.html) using lists instead of JSON.


# Building a job or pipeline using the Databricks Jobs & Pipelines user interface

For this section we will use the notebooks stored in `Databricks_workshops/Databricks workshop/04 Using Jobs & Pipelines/jobs and pipelines notebooks` to build a job using the user interface.

The job will take a catalog as a parameter. Within that catalog it will

 - Create a schema called 'shared' if it doesn't already exist (01 - Create schema if not exist)
 - Create a table called 'shared_table_runs' if it doesn't already exist (02 - Create table if not exist)
 - Insert a row into the table of the current user and the time the row was inserted (03 - Insert data)
 - Select all the rows from the table and display them (04 - Select from table)

Before following this exercise it is recommended that you look over the Notebooks in the `resources/jobs and pipelines notebooks` folder and familiarise yourself with the syntax for each step.

## Create a new job / pipeline

Right click on the 'Jobs & Pipelines' option in the left hand menu and click 'Open in a new window'.

In the Jobs & Pipelines page click the blue 'Create job' button on the right hand side.

![./../../../images/jobs_creation.png](././../../../images/jobs_creation.png "./../../../images/jobs_creation.png")

You will then be presented with a page titled 'New Job [timestamp]' which you can edit to give the job a meaningful name.

For the purposes of this exercise title the job 'test_job_[your name]'.

![./../../../images/jobs_add_task.png](././../../../images/jobs_add_task.png "./../../../images/jobs_add_task.png")

## Job settings

> The 'Job Details' pane on the right allows you to configure job level schedules and triggers, parameters which will be accessible to all tasks in the job, email notifications for successful / failed runs, and permissions on who can access and run the job. You can also add tags and descriptions to your job to help you keep track of them.

This job will create a schema in your modelling area titled `shared`, however due to the way permissions work on Databricks the person to create the schema will become it's owner, meaning no one else can use it. For this (and probably most) purposes we need anyone with access to the catalog to have the ability to use the schema, so the job will address this too by setting the owner to the *group* that manages the database.

#### Finding your group name

To do this we'll need to `ALTER` the `OWNER TO` the name of the group that owns the catalog. This will be done by one of the notebooks in the job, but we need to add it as a parameter so that notebook is able to set the permissions to the correct group. 

To find the group name navigate to your catalog in the 'Catalog Explorer' and then click the 'Permissions' tab. There should be a table with the headings 'Principal', 'Privilege' and 'Object'. You'll want to copy the 'Principle' name with "RWC" (Read / Write / Create) in it and add it as the value for the 'owner_group' job parameter (see below).

![./../../../images/jobs_permissions.png](././../../../images/jobs_permissions.png "./../../../images/jobs_permissions.png")

### Task Parameters

Click the '+ Add' button under 'Parameters' then add a parameter with a 'Key' of 'catalog' and a 'Value' of the name of your teams data catalog. Add a second parameter with a 'Key' of 'owner_group', and a value of the RWC (Read / Write / Create) group that manages the catalog.

![./../../../images/jobs_task_parameters.png](././../../../images/jobs_task_parameters.png "./../../../images/jobs_task_parameters.png")

### Task Notifications

Next, let's also setup a notification to tell us when the job has finished. Click the '+ Add' button under the 'Notifications' header then click the 'Add notification' button.

In the 'Select a destination' box choose 'Email address' and type in your email address ensure that the 'Success' and 'Failure' are ticked then click 'Save'. 

![./../../../images/jobs_task_notifications.png](././../../../images/jobs_task_notifications.png "./../../../images/jobs_task_notifications.png")

### Job Parameters and Notifications

Similar to the above you can create parameters and notifications for the whole Job using the sidebar.

![./../../../images/jobs_parameters_and_notifications.png](././../../../images/jobs_parameters_and_notifications.png "./../../../images/jobs_parameters_and_notifications.png")

## Defining tasks

Now we'll need to define the tasks in the main window. 

It's generally a good idea to give the name of the task a descriptive one, so for this job we'll just reuse the names of the notebooks for the task names. Task names can't have spaces in though so we'll use underscores (`_`) instead.

### Task Settings

You can see the task settings by clicking on the task.

### Create schema if not exist

Name the first task create_schema_if_not_exist. 

You can leave the 'Type' as notebook, but change the 'Source' to 'Workspace' (if using a git repo you want to change this to "Git provider", set up the URL, and have the path to the notebook from the directory specified), then click the blue 'Edit' link to the right.

![./../../../images/jobs_git_selection.png](././../../../images/jobs_git_selection.png "./../../../images/jobs_git_selection.png")

![./../../../images/jobs_git_configuration.png](././../../../images/jobs_git_configuration.png "./../../../images/jobs_git_configuration.png")

In the 'Git information' box put the following settings and click 'Confirm':

- Git repository URL: https://github.com/dfe-analytical-services/databricks_code_learn
- Git provider: GitHub
- Git reference: main (branch)

This ensures that the job runs the code from this Git repository on the 'main' branch.

Once you've set your git settings set the 'Path' to `/jobs and pipelines notebooks/01 - Create schema if not exist`.

Change the compute to your personal cluster.

![./../../../images/jobs_cluster.png](././../../../images/jobs_cluster.png "./../../../images/jobs_cluster.png")

Click Create task.

### Create table if not exist

Click the blue '+ Add task' button and choose 'Notebook' to add the second task. 

Notice that it has assumed that this task 'Depends on' the previous task. In this case this is what we want, although in a more complex job we may want to change the dependent task, or add others. This can be done through the 'Depends on' field.

For this task the settings should be the same as the previous task with the exception of the 'Task name' and 'Path' which should be as follows:

- Task name: create_table_if_not_exists
- Path: /jobs and pipelines notebooks/02 - Create table if not exist
- Compute: _your personal cluster_

### Insert Data

Click the blue '+ Add task' button and choose 'Notebook' to add a new task. 

For this task the setting should be the same as the previous tasks with the exception of the 'Task name' and 'Path' which should be as follows:

- Task name: insert_data
- Path: /jobs and pipelines notebooks/03 - Insert data
- Compute: _your personal cluster_

### Select from table

Click the blue '+ Add task' button and choose 'Notebook' to add a new task. 

For this task the setting should be the same as the previous tasks with the exception of the 'Task name' and 'Path' which should be as follows:

- Task name: select_from_table
- Path: /jobs and pipelines notebooks/04 - Select from table
- Compute: _your personal cluster_

## Review job

Now you've added all the notebooks as tasks your job should look similar to the one below:

![./../../../images/jobs_overview.png](././../../../images/jobs_overview.png "./../../../images/jobs_overview.png")

Assuming it does, click the blue 'Run now' button in the top right hand corner of the page. This will trigger a new run of the job and you'll likely get a pop up in the top right corner with a link to 'View run'. Click the link to watch the job run.

> Note: To run the job your cluster must be running. If it isn't running when you click the 'Run now' button your cluster will begin to start up automatically however this may take a few minutes so will delay the running of the tasks.

## Review the results

Once the job has completed and (hopefully) succeeded running all the tasks click on the 'select_from_table' task box to view the notebook it ran and it's outputs.

This should output a table with the names and run-times of everyone that has built and ran this job. 

![./../../../images/jobs_results.png](././../../../images/jobs_results.png "./../../../images/jobs_results.png")


# Scripting a job in Databricks

Jobs can be constructed through the Databricks Jobs user interface (UI), however for large or complex jobs the UI can be a time consuming way to build a job. In these scenarios it is quicker and more inline with RAP principles to script your job.

For a pipeline to be built there must be scripts, queries or notebooks available to read by Databricks, either located in your workspace, or in a Git repository.

For this example we will use the notebooks stored in `/jobs and pipelines notebooks` as the tasks, and this notebook will use code (R) to create a job from them. We'll also set it up to notify us by email when the job successfully completes.

The job constructed is the same as above.

> **Note:** The default language of this notebook has been set to `R` to prevent us having to use the `%r` _magic command_ in every cell.

## Terminology

The terms 'workflow' and 'jobs' are used pretty interchangably in the UI of Databricks but 'workflow' is archaic. In the code of the `databricks` R package they are more frequently referred to as 'jobs'.

For consistency with previous exercises this notebook uses the term 'job' but the meaning of a workflow or job is the same.

A 'task' is a component of a workflow/job, and is always referred to as a 'task'.

## Required packages

In the chunk below we load the `tidyverse` package, then install the `devtools` package and load it. 

We then use `devtools::install_github()` function to install the `databricks` package, then load it.

We'll also require the `sparklyr` package and a spark connection (`sc`) to retrieve data from Databricks and bring it into R.

In [0]:
%r
library(tidyverse)

install.packages("devtools")
library(devtools)

install_github("databrickslabs/databricks-sdk-r")
library(databricks)

library(sparklyr)

sc <- spark_connect(method = "databricks")

Assuming you have generated a PAT, run the cell below to create the widget at the top of the page. Once the widget is there paste in your personal access token into the text box.

In [0]:
%r
dbutils.widgets.text("api_token", "")

We're also going to need a widget for the name of your modelling area (`catalog`) and the name of the group that owns the catalog (`owner_group`). These will be passed through to the job and so we need them accessible to the code.  

Run the chunk below and then add in the values for the `catalog` and `owner_group` widgets.

In [0]:
%r
dbutils.widgets.text("catalog","")
dbutils.widgets.text("owner_group","")

Finally, we'll add a widget for the Databricks production URL called `host`. This isn't needed from a technical point of view, we could just hard-code it into the script as it is unlikely to change. 

However, as this repository isn't access controlled and is available to the public on GitHub it would be more secure to keep the web address of the Departments data analytics platform out of the code. It also means if the URL for Databricks ever did change there would be less code maintenance.

Run the chunk below and then populate it with the URL of this current page up until (and including) the first `/`, and your token.

In [0]:
%r
dbutils.widgets.text("host","")

## Connect to Databricks through it's API

We can now connect to the API through the `databricks` package using the `databricks::DatabricksClient()` function. It requires the `host` which and your token which we just defined above. We'll store the result in a variable called `client` as we need to pass this to the other functions in the `databricks` library, the same way we would have to pass an ODBC connection variable to a database query.

In [0]:
%r
host <- dbutils.widgets.get("host")
api_token <- dbutils.widgets.get("api_token")

client <- databricks::DatabricksClient(host = host, token = api_token)

We can then use the `databricks::clustersList()` function to fetch a list of the clusters, which we can view using `display()`. To ensure we get the correct cluster though we'll also need our username(/email address) which we can get from Databricks using the `sparklyr` connection we set up earlier.

In [0]:
%r
user_name <- sdf_sql(sc, "SELECT CURRENT_USER();") %>% collect() %>% as.character()

clusters <- databricks::clustersList(client) %>% filter(single_user_name == user_name)

display(
  clusters %>% 
    select(cluster_id, single_user_name, creator_user_name, cluster_name, spark_version, cluster_cores, cluster_memory_mb)
)


The `databricks::clustersList()` function will return any clusters that you have permission to see. Sometimes a user may have access to more than a single cluster, so we'll order the clusters by `spark_version` (descending) and take only the first result, then store it's id in a variable called `cluster_id`.

> NOTE: The data returned by the function is hierarchical, and a single 'column' may contain several other columns. As the `display()` function renders a table, you'll have to select only columns that `display()` knows how to show. Generally, the columns that are at the left-most position when you run `str(clusters)` (shows the structure).

In [0]:
%r
cluster_id <- clusters %>% 
                select(cluster_id, single_user_name, spark_version) %>% 
                arrange(desc(spark_version)) %>%
                filter(row_number() == 1) %>% 
                pull(cluster_id)

## Define Jobs and tasks

Now we have our connection to Databricks API and our `cluster_id` we'll start by setting some parameters for the job. 

Firstly we'll need a `job_name`, and the paths to the Notebooks we're wanting to include in the job. We'll also need to create a unique `task_key` for each of the Notebook tasks we're going to set up.

As the notebooks that we want to turn into tasks are stored in this repository we will need to provide relative paths (to the root folder of the repository). If we were referring to notebooks stored in our workspace but not a repository we would need to use absolute paths.

In [0]:
%r
job_name <- "test job"

create_schema_key <- "create_schema"
create_schema_path <- "techskills_workshop/Databricks workshop/04 Using Jobs & Pipelines/jobs and pipelines notebooks/01 - Create schema if not exist"

create_table_key <- "create_table"
create_table_path <- "techskills_workshop/Databricks workshop/04 Using Jobs & Pipelines/jobs and pipelines notebooks/02 - Create table if not exist"

insert_data_key <- "insert_data"
insert_data_path <- "techskills_workshop/Databricks workshop/04 Using Jobs & Pipelines/jobs and pipelines notebooks/03 - Insert data"

select_from_table_key <- "select_from_table"
select_from_table_path <- "techskills_workshop/Databricks workshop/04 Using Jobs & Pipelines/jobs and pipelines notebooks/04 - Select from table"

We can then define the tasks as lists. There are many options available available for setting when creating a task, a full list of which can be found in the tasks section of the [job API documentation](https://docs.databricks.com/api/workspace/jobs). When reading this documentation any parameter that is marked as an object needs to be passed as a list (`list()`) in R, and anything marked as an array should be passed as a vector (`c()`).

For the first task we'll give it the first `task_key` we created above, and tell it to run on our existing cluster by passing the ID of our cluster to `existing_cluster_id`, we'll then specify that it is a `notebook_task` and pass that a list with the `notebook_path` and the `source` which we will set to `GIT` (as opposed to 'Workspace').

In [0]:
%r
create_schema_task <- list(
  task_key = create_schema_key,
  existing_cluster_id = cluster_id,
  notebook_task = list(
    notebook_path = create_schema_path,
    source = "GIT"
  )
)

str(create_schema_task)

For the second task we will do the same, switching to the relevant `task_key` and `notebook_path`. In addition, we'll also add a `depends_on` clause with the previous `task_key` (passed in a list), and specify it is only to `run_if` `ALL_SUCCESS`. This means that the second task won't begin processing unless all of the tasks it `depends_on` have completed successfully.

In [0]:
%r
create_table_task <- list(
  task_key = create_table_key,
  existing_cluster_id = cluster_id,
  notebook_task = list(
                    notebook_path = create_table_path,
                    source = "GIT"
                  ),
  depends_on = list(task_key = create_schema_key),
  run_if = "ALL_SUCCESS"
)

str(create_table_task)

We'll do the same for the third task, this time switching the key, and path to the relevant variables defined above, and the dependencies to those of the `create_table_task`.

In [0]:
%r
insert_data_task <- list(
  task_key = insert_data_key,
  existing_cluster_id = cluster_id,
  notebook_task = list(
                    notebook_path = insert_data_path,
                    source = "GIT"
                  ),
  depends_on = list(task_key = create_table_key),
  run_if = "ALL_SUCCESS"
)

str(insert_data_task)

Finally we'll do the same with the last task.

In [0]:
%r
select_from_table_task <- list(
  task_key = select_from_table_key,
  existing_cluster_id = cluster_id,
  notebook_task = list(
                    notebook_path = select_from_table_path,
                    source = "GIT"
                  ),
  depends_on = list(task_key = insert_data_key),
  run_if = "ALL_SUCCESS"
)

str(select_from_table_task)

## Git details

Since we are running this job from a git repository we will need to pass the git details to the job definition. Specifically the URL of the repository, the Git provider, and the branch of the repository we are wanting to build the job from.

In [0]:
%r
git_source_list <- list(
          git_url = "https://github.com/dfe-analytical-services/databricks_code_learn",
          git_provider = "gitHub",
          git_branch = "main"
        )

## Parameters

Finally, the tasks in the job have parameters, specifically the `catalog` to write data into and the `owner_group` of that catalog to ensure the permissions are set correctly for anyone in the group to modify any schemas/tables/views created.

We could add parameters to each task individually, and if the values of those parameters was different for different tasks we would need to do this. Since the `catalog` and `owner_group` values are the same for each of our tasks we can simply add the parameters at a job level. These will be accessible to all the tasks within the job.

The list of parameters is structured as a list which contains within it a list for each parameter with the values `name` and `default`. We could add our parameter values here as the `default` and therefore by default the parameters would always contain those values unless they were overridden by parameters passed at the time the job was run.

Sometimes it is preferable for a job to fail than to make changes to database objects by accident. Having default values can run the risk of making it too easy to accidentally start a job without parameters and change the wrong database. This is a design decision that should be considered when building any automated job. 

For the purpose of this tutorial, we'll leave the default values blank so that the job fails if there are no parameters passed on the basis that it removes any chance of an accidental run that could risk overwriting previous tables/data.

In [0]:
%r
parameters_list <- list(
  list(name = 'catalog', default = ''),
  list(name = 'owner_group', default = '')
)


## Create a job

Now we have both of our tasks defined we can create the job using the `databricks::jobCreate()` function. We pass it the `client` as the first argument, then the job `name` we defined. The `tasks` are passed as a list which contains each of the task lists we built above.
We'll also tell it to send us `email_notifications` by passing a list with an `on_success` value of email addresses. As our `user_name` variable defined above is the same as our email address we can re-use this variable here.

The `jobsCreate()` function returns the ID of the job we just created, so we will want to store the response in a variable called `job` so we can refer to it later.

In [0]:
%r
  job <- jobsCreate(client,
      name = job_name,
      git_source = git_source_list,#defined above
      tasks = list(
                  create_schema_task, #defined above
                  create_table_task, #defined above
                  insert_data_task, #defined above
                  select_from_table_task), #defined above
      parameters = parameters_list, #defined above
      email_notifications = list(
                              on_success = c(user_name)
                              )
)

job



> ### _Lists of lists_
>
> _A `list()` in R is used to contain any number and type of data, including other `list()`s. This makes it excellent for storing hierarchical data in one place, however it can get quite confusing quite quickly._
>
> _Sometimes it's easier to break these `lists()` up into pieces by defining them seperately, as we did above by defining the task lists separately then passing them to the `tasks` argument in the `jobsCreate()` function._
>
> _This often makes it easier to think about and construct, but certainly makes it easier to read. Consider the code below which does exactly the same thing as the code above, but is just written all at once. It's much harder to tell what's going on when trying to absorb all that information at once._

    job <- jobsCreate(client,
    name = job_name,
    git_source = list(
        git_url = "https://github.com/dfe-analytical-services/databricks_code_learn",
        git_provider = "gitHub",
        git_branch = "main"
      ),
    tasks = list(
              list(
                  task_key = create_schema_key,
                  existing_cluster_id = cluster_id,
                  notebook_task = list(
                    notebook_path = create_schema_path,
                    source = "GIT"
                  )
                ), #create_schema
                list(
                  task_key = create_table_key,
                  existing_cluster_id = cluster_id,
                  notebook_task = list(
                                    notebook_path = create_table_path,
                                    source = "GIT"
                                  ),
                  depends_on = list(task_key = create_schema_key),
                  run_if = "ALL_SUCCESS"
                ), #create_table
                list(
                  task_key = insert_data_key,
                  existing_cluster_id = cluster_id,
                  notebook_task = list(
                                    notebook_path = insert_data_path,
                                    source = "GIT"
                                  ),
                  depends_on = list(task_key = create_table_key),
                  run_if = "ALL_SUCCESS"
                ), #insert_data
                list(
                  task_key = select_from_table_key,
                  existing_cluster_id = cluster_id,
                  notebook_task = list(
                                    notebook_path = select_from_table_path,
                                    source = "GIT"
                                  ),
                  depends_on = list(task_key = insert_data_key),
                  run_if = "ALL_SUCCESS"
                )
              ), #select_from_table
    parameters = list(
                  list(name = 'catalog', default = ''),
                  list(name = 'owner_group', default = '')
                ),
    email_notifications = list(
                            on_success = c(user_name)
                            )
    )

> _We can see here that the code is getting very long, and is also more difficult to see which options relate to which list. If it weren't for being diligent with indentation here we'd have to resort to counting brackets to see what belonged where. This is especially problematic if you accidentally delete a bracket and need to work out where it was meant to go._


## Run the job through code

We can now get the ID of the job that was created and tell the API to run the job. In a new code chunk we'll store the `job_id` from the `job` variable above. We'll then use the `databricks::jobsRunNow()` function to tell it to run the job we just created by passing it the `job_id` we just stored. 

A job may be run many times and each of these runs is given an ID. To target a specific run of a job in code we'll need to store the `job_run_id` returned by the `databricks::jobsRunNow()` function.

In [0]:
%r
job_id <- job$job_id

job_run <- jobsRunNow(client, 
                      job_id = job_id)

job_run_id <- job_run$run_id
job_run_id


## Create a link for the job run

We will now use this to create links to the job and the specific run of the job we just set off.

In a new code cell, define a `job_link` by `paste0()`ing the `host` variable we passed to the `databricks::DatabricksClient()` function earlier, followed by `"job/"` followed by the `job_id` defined above.
We can then create a `job_run_link` by `paste0()`ing together the `job_link` followed by `"/runs/"` then the `job_run_id` from the previous step.
We can then output the `job_link` as text at the bottom of the cell.

In [0]:
%r
job_link <- paste0(host,"jobs/",job_id)
job_run_link <- paste0(job_link,"/runs/", job_run_id)
job_run_link

Now click on the link above and check that it links to a job run before returning to the tutorial.

Once you've clicked the link you should see a graph, but the job will have failed. The reason it failed is because we forgot to pass any parameters to the it, and since we left the default values of the parameters blank the first code chunk that referred to them threw an error. 

This is what we wanted because not only did it prevent any accidental changes being made it also failed immediately allowing us to investigate the cause. This is preferable to accidentally making changes we didn't want but looking like a successful run, which could mean we don't even notice anything has gone wrong until later in the day when we're trying to access data we just overwrote.


## Running with parameters

Now let's run the job with parameters and (hopefully) get a successful run out of it.

We'll now take the values of the `catalog` and `owner_group` from the widgets we created earlier and pass them as parameters to the `databricks::jobsRunNow()` function.

Here the argument for passing parameters is `job_parameters`, and the list is in the structure of a single list with each item being `parameter_name = 'parameter value'`. 

In [0]:
%r
catalog <- dbutils.widgets.get("catalog")
owner_group <- dbutils.widgets.get("owner_group")

job_run <- jobsRunNow(client, 
                      job_id = job_id,
                      job_parameters = list(
                          catalog = catalog,
                          owner_group = owner_group
                        ))

job_run_id <- job_run$run_id
job_run_id

We'll now output another link and check how successful the run of the job was.

In [0]:
%r
job_link <- paste0(host,"jobs/",job_id)
job_run_link <- paste0(job_link,"/runs/", job_run_id)
job_run_link

All being well you should now see a graph of tasks turning green in sequence. Once the final task has completed check your modelling area catalog and you should see a schema called `shared`, containing a table called `shared_table_runs`.

Within that table there should be one or more rows depending on how many times this job has been run in your modelling area.

## Cleaning up

You've now created a jobs with code, and each time you re-run this notebook another job with the same name will be created. As this is a tutorial which most analysts may have to follow at some point, there's a reasonable chance that we could end up with several 'test jobs' cluttering up the job page.

To avoid that we could use the `databricks::jobsDelete()` function (as below) to clean up after ourselves. All that we need to do is pass the function the `client`, and `job_id` variables from above.

However, let's look at another way to do it.

In [0]:
%r
#jobsDelete(client, job_id)

## Bulk Clean up
 
If you have been running and re-running bits of this code iteratively, there's a good chance you already have several instances of 'test job' listed under your name.

If this is the case we'll want to clean up each of these, ideally without having to manually click through the UI process for each one.

To do this, firstly call the `databricks::jobsList()` function, passing it the `client` variable, and specifying the `name` of the jobs you want to list. Then filter the list to just the jobs with a `creator_user_name` of your email address (which is still stored in the `user_name` variable defined above). 

> Note: It's highly unlikely that you would get any jobs that weren't yours due to the default Databricks permissions model, but there is the possibility that people may have shared access to their jobs with you, so to guard against accidentally interfering with someone else's work it doesn't hurt to put an explicit safeguard in.

In [0]:
%r
my_jobs <- jobsList(client, name = "test job") 

display(my_jobs %>% select(job_id, creator_user_name, run_as_user_name, created_time))

We can now loop through the individual `job_id`s contained in `my_jobs` and use the `databricks::jobsDelete()` function to remove them all at once, programmatically.

If you do only have one instance of 'test job' this loop will only run once, and will have the same effect as just deleting the job as above, but if you have multiple occurances this loop will run as many times as needed until all of the 'test jobs' under your name have been removed.

In [0]:
%r
for(job_id in my_jobs$job_id){
      jobsDelete(client, job_id)
}

In [0]:
with open("Quiz Using Jobs & Pipelines.html", "r", encoding="utf-8") as f:
    html_content = f.read()
    displayHTML(html_content)